## 0. Dépendances

Exécute cette cellule une seule fois. Si pip modifie `transformers` ou `sentence-transformers`, redémarre ensuite le kernel.


In [20]:
%pip install -q "transformers==4.49.0" "accelerate==1.3.0" "sentence-transformers==3.4.1" rank-bm25 langgraph pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports

In [21]:
import json
import os
import re
import numpy as np
import pandas as pd
import torch

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

from chroma import ma_collection


## 2. Charger `qa_dataset.json`

In [22]:
with open("qa_dataset.json", "r", encoding="utf-8") as f:
    raw_qa = json.load(f)

def parse_generated_qa(text):
    question = ""
    ground_truth = ""

    if "Question:" in text and "Ground truth:" in text:
        question = text.split("Question:", 1)[1].split("Ground truth:", 1)[0].strip()
        ground_truth = text.split("Ground truth:", 1)[1].strip()

    return question, ground_truth

qa_dataset = []

for item in raw_qa:
    if "question" in item and "ground_truth" in item:
        question = item["question"]
        ground_truth = item["ground_truth"]
    else:
        question, ground_truth = parse_generated_qa(item.get("generated_qa", ""))

    if question and ground_truth:
        qa_dataset.append({
            "question": question,
            "ground_truth": ground_truth
        })

print("Nombre de paires Q/R :", len(qa_dataset))
print("Première question :", qa_dataset[0]["question"])
print("Ground truth :", qa_dataset[0]["ground_truth"])


Nombre de paires Q/R : 19
Première question : What is the range of the <mode> subparameter that indicates UE Type B?
Ground truth : The <mode> subparameter range (0-2) indicates UE Type B.


## 3. Charger le corpus ChromaDB

In [26]:
TOTAL_CHUNKS = ma_collection.count()
BATCH_SIZE = 5000

corpus = []
corpus_ids = []

for offset in range(0, TOTAL_CHUNKS, BATCH_SIZE):

    data = ma_collection.get(
        limit=BATCH_SIZE,
        offset=offset,
        include=["documents"]
    )

    corpus.extend(data["documents"])
    corpus_ids.extend(data["ids"])

    print(f"{len(corpus)} / {TOTAL_CHUNKS}")

CORPUS_LIMIT = len(corpus)

print("Nombre total de chunks utilisés :", CORPUS_LIMIT)

5000 / 562865
10000 / 562865
15000 / 562865
20000 / 562865
25000 / 562865
30000 / 562865
35000 / 562865
40000 / 562865
45000 / 562865
50000 / 562865
55000 / 562865
60000 / 562865
65000 / 562865
70000 / 562865
75000 / 562865
80000 / 562865
85000 / 562865
90000 / 562865
95000 / 562865
100000 / 562865
105000 / 562865
110000 / 562865
115000 / 562865
120000 / 562865
125000 / 562865
130000 / 562865
135000 / 562865
140000 / 562865
145000 / 562865
150000 / 562865
155000 / 562865
160000 / 562865
165000 / 562865
170000 / 562865
175000 / 562865
180000 / 562865
185000 / 562865
190000 / 562865
195000 / 562865
200000 / 562865
205000 / 562865
210000 / 562865
215000 / 562865
220000 / 562865
225000 / 562865
230000 / 562865
235000 / 562865
240000 / 562865
245000 / 562865
250000 / 562865
255000 / 562865
260000 / 562865
265000 / 562865
270000 / 562865
275000 / 562865
280000 / 562865
285000 / 562865
290000 / 562865
295000 / 562865
300000 / 562865
305000 / 562865
310000 / 562865
315000 / 562865
320000 / 562

## 7. Retrieval hybride BM25 + BGE + RRF

## 4. BM25


In [27]:
def tokenize(text):
    return re.findall(
        r"[A-Za-z0-9]+(?:[._-][A-Za-z0-9]+)*",
        text.lower()
    )

tokenized_corpus = [tokenize(doc) for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 prêt")


BM25 prêt


## 5. BGE


In [28]:
BGE_MODEL = "BAAI/bge-small-en-v1.5"
BGE_CACHE = f"bge_embeddings_{CORPUS_LIMIT}.npy"
device = "cuda" if torch.cuda.is_available() else "cpu"

bge_model = SentenceTransformer(BGE_MODEL , device = device )

if os.path.exists(BGE_CACHE):
    bge_embeddings = np.load(BGE_CACHE)
    print("Embeddings BGE chargés depuis le cache")
else:
    bge_embeddings = bge_model.encode(
        corpus,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    np.save(BGE_CACHE, bge_embeddings)
    print("Embeddings BGE calculés et sauvegardés")

print("Shape :", bge_embeddings.shape)


Batches:   0%|          | 46/17590 [08:29<53:57:29, 11.07s/it]


KeyboardInterrupt: 

## 6. Fusion RRF


In [ ]:
def rrf_fusion(bm25_indices, bge_indices, k=60, top_k=5):
    scores = {}

    for rank, idx in enumerate(bm25_indices, start=1):
        idx = int(idx)
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)

    for rank, idx in enumerate(bge_indices, start=1):
        idx = int(idx)
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)

    ranking_final = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return ranking_final[:top_k]


In [ ]:
def hybrid_retrieve(question, top_k=3):
    # BM25
    bm25_scores = bm25.get_scores(tokenize(question))
    top_bm25 = np.argsort(bm25_scores)[::-1][:10]

    # BGE
    query_embedding = bge_model.encode(
        question,
        normalize_embeddings=True
    )
    bge_scores = bge_embeddings @ query_embedding
    top_bge = np.argsort(bge_scores)[::-1][:10]

    # RRF
    fused = rrf_fusion(top_bm25, top_bge, top_k=top_k)

    indices = [idx for idx, score in fused]
    chunks = [corpus[idx] for idx in indices]
    ids = [corpus_ids[idx] for idx in indices]

    return chunks, ids


## 8. Charger `microsoft/Phi-4-mini-instruct`

In [ ]:
MODEL_NAME = "microsoft/Phi-4-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=False
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=False
)

print("Phi-4 chargé")
print("Device :", model.device)


Loading checkpoint shards: 100%|██████████| 2/2 [00:19<00:00,  9.57s/it]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


Phi-4 chargé
Device : cuda:0


## 9. Fonction de génération Phi-4

In [ ]:
def generate_phi4(user_prompt):
    messages = [
        {
            "role": "system",
            "content": "You are a technical assistant specialized in 3GPP specifications."
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()


## 10. LangGraph

Pipeline :

`Question → Phi-4 sans RAG → Retrieval hybride → Phi-4 avec RAG → résultat`


In [ ]:
class EvaluationState(TypedDict):
    question: str
    ground_truth: str
    without_rag: str
    retrieved_context: str
    retrieved_ids: list
    with_rag: str


## 11. Nodes

In [ ]:
def baseline_node(state):
    prompt = f"""
Answer the following 3GPP technical question.

Question:
{state["question"]}

Answer:
"""

    answer = generate_phi4(prompt)

    return {"without_rag": answer}


def retrieval_node(state):
    chunks, ids = hybrid_retrieve(
        state["question"],
        top_k=3
    )

    context = "\n\n---\n\n".join(chunks)

    return {
        "retrieved_context": context,
        "retrieved_ids": ids
    }


def rag_node(state):
    prompt = f"""
Answer the technical question using only the provided 3GPP context.

If the answer is not contained in the context, say that the context does not provide enough information.

Context:
{state["retrieved_context"]}

Question:
{state["question"]}

Answer:
"""

    answer = generate_phi4(prompt)

    return {"with_rag": answer}


## 12. Construire le graphe

In [ ]:
builder = StateGraph(EvaluationState)

builder.add_node("baseline", baseline_node)
builder.add_node("retrieval", retrieval_node)
builder.add_node("rag", rag_node)

builder.add_edge(START, "baseline")
builder.add_edge("baseline", "retrieval")
builder.add_edge("retrieval", "rag")
builder.add_edge("rag", END)

evaluation_graph = builder.compile()

print("LangGraph prêt")


LangGraph prêt


## 13. Test sur une seule question

In [ ]:
test_item = qa_dataset[0]

test_result = evaluation_graph.invoke({
    "question": test_item["question"],
    "ground_truth": test_item["ground_truth"],
    "without_rag": "",
    "retrieved_context": "",
    "retrieved_ids": [],
    "with_rag": ""
})

print("QUESTION")
print(test_result["question"])

print("\n1. PHI-4 SANS RAG")
print(test_result["without_rag"])

print("\n2. PHI-4 + RAG")
print(test_result["with_rag"])

print("\n3. GROUND TRUTH")
print(test_result["ground_truth"])


QUESTION
What is the range of the <mode> subparameter that indicates UE Type B?

1. PHI-4 SANS RAG
In the context of 3GPP specifications, the <mode> subparameter that indicates UE Type B (User Equipment Type B) is typically associated with the "cellular" mode. The range of the <mode> subparameter for UE Type B is usually 0, which corresponds to the cellular mode. This is in contrast to other modes such as "p2p" (peer-to-peer) or "g2g" (gateway-to-gateway), which have different ranges. For example, "p2p" might be represented by a range of 1, and "g2g" might be represented by a range of 2. However, the exact ranges can vary based on the specific 3GPP specification being referenced. Always refer to the relevant 3GPP technical document for precise information.

2. PHI-4 + RAG
The context provided does not contain enough information to determine the range of the <mode> subparameter that indicates UE Type B.

3. GROUND TRUTH
The <mode> subparameter range (0-2) indicates UE Type B.


## 14. Lancer les 20 questions

Cette partie fait 2 générations Phi-4 par question. Sur CPU, cela peut prendre du temps.


In [ ]:
N_QUESTIONS = 5

results = []

for i, item in enumerate(qa_dataset[:N_QUESTIONS], start=1):

    result = evaluation_graph.invoke({
        "question": item["question"],
        "ground_truth": item["ground_truth"],
        "without_rag": "",
        "retrieved_context": "",
        "retrieved_ids": [],
        "with_rag": ""
    })

    results.append({
        "question": result["question"],
        "llm_without_rag": result["without_rag"],
        "llm_with_rag": result["with_rag"],
        "ground_truth": result["ground_truth"],
        "retrieved_ids": result["retrieved_ids"]
    })

    print(f"Question {i}/{N_QUESTIONS} terminée")


Question 1/5 terminée
Question 2/5 terminée
Question 3/5 terminée
Question 4/5 terminée
Question 5/5 terminée


## 15. Tableau final

In [ ]:
results_df = pd.DataFrame(results)

display(
    results_df[
        [
            "question",
            "llm_without_rag",
            "llm_with_rag",
            "ground_truth"
        ]
    ]
)


,question,llm_without_rag,llm_with_rag,ground_truth
0,What is the range of the <mode> subparameter t...,"In the context of 3GPP specifications, the <mo...",The context provided does not contain enough i...,The <mode> subparameter range (0-2) indicates ...
1,For the UE that supports the Sul operation giv...,For the UE (User Equipment) that supports the ...,The context provided does not contain enough i...,Only the listed test points in Table 7.3C.2-4 ...
2,What is the schema type of the request body fo...,The schema type of the request body for the 'p...,The context provided does not contain any info...,The schema type of the request body for the 'p...
3,What is the bandwidth combination indicated in...,The bandwidth combination indicated in the row...,The context provided does not contain enough i...,20+20+20MHz
4,How does a ds-tt determine the announce interv...,In the context of the 3GPP TS 29.060 (GTP Prot...,The provided context does not contain informat...,"According to the excerpt, a ds-tt determines t..."


## 16. Sauvegarder les résultats

In [ ]:
results_df.to_csv(
    "evaluation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

with open("evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=4
    )

print("Fichiers créés :")
print("- evaluation_results.csv")
print("- evaluation_results.json")


Fichiers créés :
- evaluation_results.csv
- evaluation_results.json


## Sortie finale attendue

Pour chaque question :
- **LLM completion without RAG**
- **LLM + RAG completion**
- **Ground truth completion**

Le chemin RAG est orchestré avec LangGraph et suit :

**BM25 + BGE → RRF → contexte → Phi-4**
